# Appendix A: CM centers, scales, and the first 20 coefficients
For *Recurrence formulas for Taylor expansions at CM points on genus-one and genus-two Shimura curves*.
Choose the **SageMath** kernel and **Restart Kernel and Run All**. All inputs and
functions are contained in this notebook; no other package files are read.

The appendix convention is
$$f_N(z)=(1-w_N)^2\sum_{n\ge0}\frac{c_{N,n}}{n!}(\Theta_Nw_N)^n,
\qquad w_N=\frac{z-\tau_N}{z-\overline{\tau_N}}.$$
We check the nine exact CM centers, compute all twelve scales from Nelson sums,
and generate coefficients by the coupled recurrence. `first_coefficients(form,N)`
returns exactly $c_0,\ldots,c_{N-1}$; the displayed tables also give $c_n/n!$.
The recurrence data and algebraic scale recognitions remain the stated hypotheses.
Numerical comparisons measure observed agreement, not rigorous infinite-tail bounds.

## Explicit quaternion orders and CM embeddings
For each level, the data are $(a,b,D,R,\mu)$ in the quaternion algebra
$(a,b/\mathbb Q)$ with basis $(1,i,j,ij)$. Rows of $R$ are the chosen order basis.
Its norm-one units give the arithmetic group used for the CM expansion.
The code checks the identity, multiplication closure, integrality, and
reduced discriminant $N$, proving maximality of this explicit order.
It then checks the CM discriminant/class number and solves the fixed-point
equation for $\mu$ in the upper half-plane. No group generators or Magma are needed.
The order and embedding choices fix the normalization.

In [1]:
CM_DATA = {
    14: (7, -2, -11,
         ['1/2 1/2 0 1/2', '0 1 0 0', '0 0 1/2 1/2', '0 0 0 1'], "1/2 -1/2 -3/2 0"),
    15: (15, -30, -7,
         ['1/2 0 1/4 1/20', '0 1/2 1/4 1/60', '0 0 1/2 1/30', '0 0 0 1/15'], "1/2 0 -1/4 1/60"),
    21: (21, -1, -43,
         ['1/2 1/2 0 0', '0 1 0 0', '0 0 1/2 1/2', '0 0 0 1'], "1/2 -1/2 -10 -2"),
    33: (33, -1, -67,
         ['1/2 1/2 0 0', '0 1 0 0', '0 0 1/2 1/2', '0 0 0 1'], "1/2 -1/2 -5 0"),
    34: (34, -3, -11,
         ['1/2 0 1/6 1/3', '0 1/2 0 1/2', '0 0 1/3 2/3', '0 0 0 1'], "1/2 1/2 -7/2 1/2"),
    46: (46, -3, -163,
         ['1/2 0 1/6 1/3', '0 1/2 0 1/2', '0 0 1/3 2/3', '0 0 0 1'], "1/2 0 -35/6 -2/3"),
    26: (26, -5, -11,
         ['1/2 0 1/10 1/10', '0 1/2 0 1/2', '0 0 1/5 1/5', '0 0 0 1'], "1/2 1 -7/2 -1/2"),
    38: (-19, 2, -11,
         ['1/2 1/2 0 0', '0 1 0 0', '0 0 1/2 1/2', '0 0 0 1'], "1/2 -1/2 -1 0"),
    58: (-29, 2, -11,
         ['1/2 1/2 0 1/2', '0 1 0 0', '0 0 1/2 1/2', '0 0 0 1'], "1/2 1/2 3/2 0"),
}
CM_DATA = {N: (a,b,D,[r.split() for r in rows],mu.split())
           for N,(a,b,D,rows,mu) in CM_DATA.items()}

In [2]:
from sage.all import *
from collections import defaultdict
from itertools import product
from math import isqrt, cos, exp, log, gamma, pi

RF = RealField(336)                 # about 100 decimal digits
CF = ComplexField(336)

def lattice(rows):
    """A row basis of the exact Z-span, in upper triangular Hermite form."""
    M = matrix(QQ, rows)
    d = lcm([x.denominator() for x in M.list()])
    return (d*M).change_ring(ZZ).hermite_form(include_zero_rows=False)/d

def embedding(a, b, q):
    x, y, z, w = q
    if a < 0:
        a, b, y, z, w = b, a, z, y, -w
    p, s = QQbar(a).sqrt(), QQbar(-b).sqrt()
    return matrix(QQbar, [[x+y*p, (z+w*p)*s], [(-z+w*p)*s, x-y*p]])

class CMPoint:
    """Exact order/CM inputs and the split lattices in Nelson's formula."""
    def __init__(self, N):
        a, b, D, basis, mu = CM_DATA[N]
        self.N, self.D = N, D
        B = QuaternionAlgebra(QQ, a, b)
        R, mu = matrix(QQ, basis), B(list(map(QQ, mu)))
        inside = lambda q: all(x in ZZ for x in vector(QQ, list(q))*R.inverse())
        rb = [B(list(row)) for row in R]
        assert B.discriminant() == N and inside(B(1)) and inside(mu)
        assert all(inside(x*y) for x in rb for y in rb)
        assert all(q.reduced_trace() in ZZ and q.reduced_norm() in ZZ for q in rb)
        T = matrix(QQ, [[(x*y.conjugate()).reduced_trace() for y in rb] for x in rb])
        assert abs(T.det()) == N*N  # an integral order of this discriminant is maximal
        assert mu.reduced_trace() == 1 and 1-4*mu.reduced_norm() == D
        assert QuadraticField(D).discriminant() == D and QuadraticField(D).class_number() == 1
        M = embedding(a, b, mu)
        sign = 1 if M[1,0].real() > 0 else -1
        self.tau = (M[0,0]-M[1,1]+sign*QQbar(D).sqrt())/(2*M[1,0])
        assert self.tau.imag() > 0
        assert M[1,0]*self.tau**2+(M[1,1]-M[0,0])*self.tau-M[0,1] == 0
        r = 2*mu-1
        candidates = [B([0]+list(v)) for v in product(range(-3,4), repeat=3) if any(v)]
        candidates = [v for v in candidates if (v*mu.conjugate()).reduced_trace()==0]
        if not candidates:
            v = (-a*mu[1],-b*mu[2],a*b*mu[3])
            candidates = [q for q in [B([0,-v[1],v[0],0]),B([0,-v[2],0,v[0]]),
                                      B([0,0,-v[2],v[1]])] if q]
            candidates = candidates[:1]  # exact fallback used in the supplied evaluator
        j = min(candidates,key=lambda v: abs(v.reduced_norm()))
        self.c = -j.reduced_norm()
        assert self.c > 0 and j*r == -r*j
        C = matrix(QQ, [list(q) for q in [B(1),r,j,r*j]]).inverse()
        x, y = CF(self.tau).real(), CF(self.tau).imag()
        sigma = matrix(CF, [[y.sqrt(), x/y.sqrt()], [0,1/y.sqrt()]])
        def XY(q):
            U = sigma.inverse()*embedding(a,b,q).change_ring(CF)*sigma
            return ((U[0,0]+U[1,1]+CF.gen()*(U[0,1]-U[1,0]))/2,
                    (U[0,0]-U[1,1]+CF.gen()*(U[0,1]+U[1,0]))/2)
        Xr, _ = XY(r)
        self.eps = 1 if Xr.imag()>0 else -1
        Xj, self.Y = XY(j)
        assert abs(Xj)<RF('1e-90') and abs(abs(self.Y)**2-self.c)<RF('1e-90')
        dual = T.inverse()*R
        self.split = {}
        for d in divisors(N):
            V = lattice(list(R.rows())+list(((N//d)*dual).rows()))*C
            den = lcm([x.denominator() for x in V.list()])
            H = (den*V).change_ring(ZZ).hermite_form()
            self.split[int(d)] = (int(den), [list(map(int,row)) for row in H])

    def grouped_sums(self, d, an, cutoff, nmax):
        """Exact integer-pair sums in Z[r], r^2=D; cutoff bounds |kappa_1|^2."""
        L, H = self.split[d]
        D, p = int(self.D), -int(self.D)
        cn, cd = int(self.c.numerator()), int(self.c.denominator())
        (h11,h12,h13,h14),(_,h22,h23,h24),(_,_,h33,h34),(_,_,_,h44) = H
        bound = int(cutoff)*L*L
        def powers(x,y,start,count):
            u,v = 1,0
            for n in range(start+count):
                if n>=start: yield (u,v)
                u,v = u*x+D*v*y,u*y+v*x
        pairs = lambda: [[0,0] for _ in range(nmax+1)]
        classes = defaultdict(lambda: defaultdict(pairs))
        for a in range(-isqrt(bound)//h11-1, isqrt(bound)//h11+2):
            x = a*h11
            if x*x>bound: continue
            t = isqrt((bound-x*x)//p)+1
            for b in range((-t-a*h12)//h22-1, (t-a*h12)//h22+2):
                y = a*h12+b*h22
                k = x*x+p*y*y
                if not 0<k<=bound: continue
                s,t = a*h13+b*h23,a*h14+b*h24
                q = s//h33
                acc = classes[(s-q*h33,(t-q*h34)%h44)][k]
                for n,(u,v) in enumerate(powers(x,y,2,nmax+1)):
                    acc[n][0]+=u; acc[n][1]+=v
        G = defaultdict(pairs)
        for (s,t),rows in classes.items():
            bound2 = (cd*max(rows)-1)//cn
            second = defaultdict(pairs)
            for a in range(-isqrt(bound2)//h33-2, isqrt(bound2)//h33+3):
                x = s+a*h33
                if x*x>bound2: continue
                bnd = isqrt((bound2-x*x)//p)+1
                base = t+a*h34
                for b in range((-bnd-base)//h44-1,(bnd-base)//h44+2):
                    y = base+b*h44
                    k = x*x+p*y*y
                    if k>bound2: continue
                    acc = second[k]
                    for n,(u,v) in enumerate(powers(x,-y,0,nmax+1)):
                        acc[n][0]+=u; acc[n][1]+=v
            for k,J in rows.items():
                I = pairs()
                for l in sorted(second):
                    norm = cd*k-cn*l
                    if norm<=0: break
                    index,rem = divmod(d*norm,cd*L*L)
                    assert rem==0
                    for n,(u,v) in enumerate(second[l]):
                        I[n][0]+=an[index]*u; I[n][1]+=an[index]*v
                for n,((a,b),(c,e)) in enumerate(zip(J,I)):
                    G[k][n][0]+=a*c+D*b*e; G[k][n][1]+=a*e+b*c
        return L,G

In [3]:
from IPython.display import display, Math, Markdown
tau_expected = {
    14: (-sqrt(14)+sqrt(-22))/6,
    15: (sqrt(-210)-sqrt(-14))/14,
    21: (-21-5*sqrt(21)+5*sqrt(-43)+sqrt(-903))/16,
    33: (-sqrt(33)+sqrt(-67))/10,
    34: (-34*sqrt(3)+7*sqrt(102)+7*sqrt(-33)-sqrt(-1122))/45,
    46: (35*sqrt(-489)+4*sqrt(-22494))/489,
    26: (52*sqrt(5)+14*sqrt(130)+7*sqrt(-55)+sqrt(-1430))/115,
    38: (-2*sqrt(38)+sqrt(-209))/19,
    58: (-3*sqrt(58)+sqrt(-319))/29,
}
points = {N: CMPoint(N) for N in CM_DATA}
for N, point in points.items():
    assert point.tau == QQbar(tau_expected[N]), N
    display(Math(r'\tau_{'+str(N)+r'}='+latex(tau_expected[N])))
print('PASS: all nine maximal-order, CM, fixed-point, and appendix-radical checks.')

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

PASS: all nine maximal-order, CM, fixed-point, and appendix-radical checks.


## Compute $\Theta$ from finite Nelson sums
The next cell implements the weight integral and the finite lattice sums in
[Nelson's method, Â§3](https://arxiv.org/pdf/1210.1243). Arithmetic in the lattices
and grouped quadratic-field powers is exact. The integral and final sums use
336-bit real/complex arithmetic (about 100 decimal digits), with an 80-digit
quadrature target and squared lattice cutoffs 144 and 196.

Only derivative indices 0 and 1 are needed for $\Theta$. The companion Nelson
notebook uses the higher derivatives needed for the initial $c_n$. The period is
computed from the class-number-one Gamma product
$$\pi\Omega_D^2=\frac1{2|D|}\prod_{j=1}^{|D|-1}
\Gamma(j/|D|)^{(D/j)}.$$
The weight step and tail rules are numerical heuristics, so agreement at two
cutoffs is not an analytic error enclosure.

In [4]:
def weight(x, ks, digits=80):
    """Trapezoidal evaluation of Nelson's Bessel-weight integral.

    The same step/tail heuristic as the supplied evaluator; no interval bound.
    Li_{-s}(q) = sum_k k! S(s+1,k+1) (q/(1-q))^(k+1).
    """
    n,x0 = max(ks),float(x)
    xa = x0*cos(1.5); rho = exp(-xa/2)
    majorant = log(2*gamma(n)*2**(2*n-1))-n*log(xa)+log(x0*rho/(1-rho)**2+rho/(1-rho))
    majorant = min(majorant,log(2*(x0+1))+0.5*log(2*pi/xa)+n*n/(2*xa)-xa+30)
    log_eps = -digits*log(10)-x0
    h = RF(str(2*pi*1.5/(majorant-log_eps+2)))
    end = 0.0
    while log(4/3*(x0+1))+n*end-x0*float(cosh(end))>log_eps-10 or x0*float(cosh(end))<30:
        end+=0.02
    coeff = {s:[factorial(k)*stirling_number2(s+1,k+1) for k in range(s+1)]
             for s in range(min(ks),n+2)}
    total = {k:RF(0) for k in ks}
    for j in range(int(end/float(h))+3):
        e = (j*h).exp(); ei = 1/e
        q = (-x*(e+ei)/2).exp(); r = q/(1-q)
        powers = [r**k for k in range(n+3)]
        Li = {s:sum(c*powers[k+1] for k,c in enumerate(coeff[s])) for s in coeff}
        ch = [(e**k+ei**k)/2 for k in range(n+1)]
        for k in ks:
            total[k]+=(RF(1)/2 if j==0 else 1)*(x*ch[k-1]*Li[k+1]-ch[k]*Li[k])
    return {k:total[k]*h/2**(k-1) for k in ks}

def nelson(form, point, nmax=1, cutoff=196):
    """Return normalized c_2,...,c_nmax and Theta from finite Nelson sums."""
    N = point.N
    an = list(map(int,EllipticCurve(form+'1').anlist(int(N*cutoff+10))))
    sums = [CF(0) for _ in range(nmax+1)]
    W = {}
    for d in divisors(N):
        L,G = point.grouped_sums(int(d),an,int(cutoff),int(nmax))
        for k,terms in G.items():
            m = QQ(k)/(L*L)
            if m not in W: W[m]=weight(4*RF.pi()*RF(m).sqrt(),range(2,nmax+3))
            for n,(a,b) in enumerate(terms):
                term = CF(a,point.eps*b*RF(-point.D).sqrt())
                sums[n]+=term*W[m][n+2]/(d*an[d]*k*RF(L)**n*RF(k).sqrt()**n)
    assert sums[0] and sums[1]
    c = {n:sums[n]*sums[0]**(n-1)/sums[1]**n for n in range(2,nmax+1)}
    theta = -4*RF.pi()*point.Y*sums[1]/sums[0]
    return c,theta

def period(D):
    """pi*Omega_D^2 from the Gamma-product definition (class number one)."""
    d = abs(D)
    return prod((RF(j)/d).gamma()**kronecker_symbol(D,j) for j in range(1,d))/(2*d)

In [5]:
alpha = {
    "14a": 1+sqrt(-77)/7,
    "15a": -8*sqrt(15)/15,
    "21a": -63/4-3*sqrt(-903)/4,
    "33a": -23/5-23*sqrt(-2211)/165,
    "34a": 1-11*sqrt(34)/102-sqrt(-11)/3-sqrt(-374)/34,
    "46a": 7855*sqrt(46)/69,
    "26a": -21*(988-33*sqrt(26)-156*sqrt(-11)-19*sqrt(-286))/2990,
    "26b": (988-33*sqrt(26)-156*sqrt(-11)-19*sqrt(-286))/2990,
    "38a": -9*(4+sqrt(-22))/19,
    "38b": 13*(4+sqrt(-22))/19,
    "58a": -5*(6+sqrt(-22))/29,
    "58b": 3*(6+sqrt(-22))/29,
}
periods = {D: period(D) for D in {point.D for point in points.values()}}
theta_results = {}
for form, value in alpha.items():
    point = points[int(form[:-1])]
    P = periods[point.D]
    _, low = nelson(form,point,nmax=1,cutoff=144)
    _, high = nelson(form,point,nmax=1,cutoff=196)
    target = CF(QQbar(value))*P
    observed = abs(high-target)
    normalized = abs(high/P-CF(QQbar(value)))
    change = abs(high-low)
    assert max(abs(low-target),observed,normalized,change) < RF('1e-40'), form
    theta_results[form] = (high,observed,normalized,change)
    display(Math(r'\Theta_{'+form+r'}/(\pi\Omega_D^2)='+latex(value)))
    print(form, ': Theta =', high.n(digits=30))
    print('  observed scale difference:', observed.n(digits=6),
          '; normalized difference:', normalized.n(digits=6),
          '; cutoff change:', change.n(digits=6), flush=True)
print('PASS: all twelve scales and cutoff comparisons are below 1e-40.')

<IPython.core.display.Math object>

14a : Theta = 0.553368343038086157248839792541 + 0.693683929038484764190287786194*I
  observed scale difference: 7.59235e-69 ; normalized difference: 1.37202e-68 ; cutoff change: 1.41282e-58


<IPython.core.display.Math object>

15a : Theta = -1.62550112495823548562541731433 + 7.89460505012000347046944351285e-155*I
  observed scale difference: 2.61336e-69 ; normalized difference: 3.32090e-69 ; cutoff change: 3.13711e-58


<IPython.core.display.Math object>

21a : Theta = -1.59696263706282979215367337783 - 2.28517432449037316530353877564*I
  observed scale difference: 2.57504e-68 ; normalized difference: 2.53963e-67 ; cutoff change: 7.61946e-58


... (15 outputs left out)

58a : Theta = -0.572450010039399473016041164698 - 0.447504758133313292651427988894*I
  observed scale difference: 6.37775e-69 ; normalized difference: 1.15253e-68 ; cutoff change: 1.15673e-58


<IPython.core.display.Math object>

58b : Theta = 0.343470006023639683809624698819 + 0.268502854879987975590856793336*I
  observed scale difference: 3.73015e-69 ; normalized difference: 6.74080e-69 ; cutoff change: 6.94124e-59


PASS: all twelve scales and cutoff comparisons are below 1e-40.


## Recurrence input and exact coefficient generation
Each row below is $(H,\mathcal R,L,u_0,u_1)$. Sections 4.1â€“4.2 recover these data;
they are written here to make the notebook independent. We solve
$$A''=\frac{3(A')^2}{2A}-L\mathcal R(u)A^3,\qquad
u''=\frac{A'}A u'+\frac L2 A^2H'(u),$$
with $A(0)=A'(0)=1$. At degree $n$, divide the right-hand coefficient by
$(n+1)(n+2)$ to obtain the coefficient of $q^{n+2}$. Computation takes place in
$\mathbb Q[[q]]$; the returned $c_n$ multiply ordinary coefficients by $n!$.

In [6]:
Pu = PolynomialRing(QQ,'u'); u = Pu.gen(); Fu = Pu.fraction_field()
recurrence = {
    "14a": (Pu([0, -512, 52, -4]), Fu(3*(4-u)/8),
             QQ("-11/9"), QQ("16"), QQ("-352/3")),
    "15a": (Pu([0, -108, -328, -12]), Fu(-4-4*u/3),
             QQ("-7/64"), QQ("9"), QQ("-63")),
    "21a": (Pu([0, -1372, 376, -28]), Fu(-16*(17*u**2+392*u-2401)/(u-49)**2),
             QQ("-43/3969"), QQ("441/25"), QQ("-9632/375")),
    "33a": (Pu([0, -972, -40, -12]), Fu(-4*(u**3+49*u**2+19*u+27)/(u*(u-1)**2)),
             QQ("-67/529"), QQ("9/25"), QQ("-19296/2875")),
    "34a": (Pu([-236, 104, -71, 26, -3]), Fu((279*u**2+3104*u-39264)/(2*(15*u-16)**2)),
             QQ("-11/25"), QQ("-15/4"), QQ("-3179/80")),
    "46a": (Pu([0, -2048, 180, -4]), Fu(-(3*u**3+52*u**2+13824*u-311296)/(8*(u-64)**2)),
             QQ("-844992/61701025"), QQ("3136/81"), QQ("-15188992/636255")),
    "26a": (Pu([-676, -96, 76, -8]), Fu(9/2-9/u-507/(2*u**2)),
             QQ("-11/1764"), QQ("1"), QQ("44/21")),
    "26b": (Pu([-8, 76, -96, -676]), Fu(3+14/u-3/u**2),
             QQ("-11/4"), QQ("1"), QQ("44")),
    "38a": (Pu([-76, -328, -236, -64]), Fu(-6*u-45/2-57/u-57/(2*u**2)),
             QQ("-11/324"), QQ("1"), QQ("44/9")),
    "38b": (Pu([-64, -236, -328, -76]), Fu(-16-52/u-30/u**2),
             QQ("-11/676"), QQ("1"), QQ("44/13")),
    "58a": (Pu([-6728, -3448, -312, -8]), Fu(5-144358/(243*u)-2523/u**2-229376/(243*(u+27))+65536/(9*(u+27)**2)),
             QQ("121/400"), QQ("-11"), QQ("-176/5")),
    "58b": (Pu([-8, -312, -3448, -6728]), Fu(-39625/243-34/u-3/u**2-163840/(243*(27*u+1))-65536/(243*(27*u+1)**2)),
             QQ("-11/144"), QQ("-1/11"), QQ("-16/33")),
}

In [7]:
from operator import index

def recurrence_series(form, N):
    """Return A(q),u(q) through degree N-1, starting only from c0=c1=1."""
    if isinstance(N, bool):
        raise ValueError('N must be a nonnegative integer')
    N = index(N)
    if N < 0:
        raise ValueError('N must be nonnegative')
    H,R,L,u0,u1 = recurrence[form]
    S = PowerSeriesRing(QQ,'q',default_prec=max(N+2,4))
    aa,uu = [QQ(1),QQ(1)], [u0,u1]
    for n in range(max(0,N-2)):
        A,U = S(aa),S(uu)
        R_U = R.numerator()(U)/R.denominator()(U)
        rhsA = QQ(3)/2*A.derivative()**2/A-L*R_U*A**3
        rhsU = A.derivative()/A*U.derivative()+L/2*A**2*H.derivative()(U)
        aa.append(QQ(rhsA[n])/((n+1)*(n+2)))
        uu.append(QQ(rhsU[n])/((n+1)*(n+2)))
    return S(aa[:N]).add_bigoh(N),S(uu[:N]).add_bigoh(N)

def first_coefficients(form, N):
    """Exponential coefficients c_0,...,c_(N-1); examples: ('46a',20), ('58b',40)."""
    A,U = recurrence_series(form,N)
    return [QQ(factorial(n)*A[n]) for n in range(index(N))]

assert first_coefficients('14a',0) == []
assert first_coefficients('14a',1) == [1]
assert first_coefficients('14a',2) == [1,1]

In [8]:
printed = {
    "14a": "1 1 -4 214/9 24",
    "15a": "1 1 -1/4 27/16 -381/64",
    "21a": "1 1 -905/3969 -1318/27783 -4357004/5250987",
    "33a": "1 1 -72337/529 9202302/12167 3009372468/279841 -1149714703920/6436343 176790432531276/148035889",
    "34a": "1 1 -12/25 2762/125 -17736/625 -67648/125 6553872/15625",
    "46a": "1 1 27824049/123402050 -114816966372/484661551375 -784666539922896/3807016486050625 -3694325348311426944/5980822899585531875 939622156860406531404672/234896819381221764390625",
    "26a": "1 1 -16/147 274/3087 -2488/9261 352768/1361367 5961776/28588707",
    "26b": "1 1 40 -734 -11240 1308544 5134480",
    "38a": "1 1 -64/27 -62/27 -19672/2187 41984/6561 -38071696/177147",
    "38b": "1 1 -16/169 -878/2197 7688/28561 -2699648/371293 25568336/4826809",
    "58a": "1 1 -4/5 314/125 31192/625 -1977728/625 4428752/3125 -4829523664/78125",
    "58b": "1 1 140/9 262/3 -39592/81 420224/81 -303950320/729",
}
for form,(H,R,L,u0,u1) in recurrence.items():
    assert u1**2 == L*H(u0) and R.denominator()(u0) != 0
    A,U = recurrence_series(form,32)
    R_U = R.numerator()(U)/R.denominator()(U)
    rA = A*A.derivative(2)-QQ(3)/2*A.derivative()**2+L*R_U*A**4
    rU = 2*A*U.derivative(2)-2*A.derivative()*U.derivative()-L*A**3*H.derivative()(U)
    first = U.derivative()**2-L*A**2*H(U)
    assert min(rA.prec(),rU.prec(),first.prec()) >= 30
    assert all(rA[n] == rU[n] == first[n] == 0 for n in range(30)), form
    c = first_coefficients(form,32)
    expected = [QQ(x) for x in printed[form].split()]
    assert c[:len(expected)] == expected, form
    display(Markdown('### '+form+' â€” coefficients 0 through 19'))
    rows = ['| n | c_n | c_n/n! |','|---:|---:|---:|']
    rows += ['| '+str(n)+' | '+str(c[n])+' | '+str(c[n]/factorial(n))+' |' for n in range(20)]
    display(Markdown('\n'.join(rows)))
print('PASS: all twelve printed expansions; 32 coefficients computed and 30 residual coefficients checked per form.')

### 14a â€” coefficients 0 through 19

| n | c_n | c_n/n! |
|---:|---:|---:|
| 0 | 1 | 1 |
| 1 | 1 | 1 |
| 2 | -4 | -2 |
| 3 | 214/9 | 107/27 |
| 4 | 24 | 1 |
| 5 | 47104/81 | 5888/1215 |
| 6 | -163376/81 | -10211/3645 |
| 7 | -25763024/729 | -230027/32805 |
| 8 | -359785216/729 | -401546/32805 |
| 9 | -11587769984/6561 | -12932779/2657205 |
| 10 | 55550801152/2187 | 30999331/4428675 |
| 11 | -5280560316416/59049 | -267885568/119574225 |
| 12 | -368115980739584/59049 | -4668678733/358722675 |
| 13 | 49687173291019264/531441 | 48474156011/3228504075 |
| 14 | -248659593004404736/531441 | -225261255838/41970552975 |
| 15 | -336598664382284294144/4782969 | -304925448224327/5666024651625 |
| 16 | 226213608266292297728/1594323 | 12807961898339/1888674883875 |
| 17 | -83335836960457641426944/43046721 | -277551884214016/50994221864625 |
| 18 | -9364169257051756027510784/43046721 | -265094403411688471/7802115945287625 |
| 19 | -441115888869696280127799296/387420489 | -657249711979287871/70219043507588625 |

### 15a â€” coefficients 0 through 19

| n | c_n | c_n/n! |
|---:|---:|---:|
| 0 | 1 | 1 |
| 1 | 1 | 1 |
| 2 | -1/4 | -1/8 |
| 3 | 27/16 | 9/32 |
| 4 | -381/64 | -127/512 |
| 5 | -10725/256 | -715/2048 |
| 6 | 166635/1024 | 3703/16384 |
| 7 | -1040445/4096 | -3303/65536 |
| 8 | 16448355/16384 | 52217/2097152 |
| 9 | -63569205/65536 | -22423/8388608 |
| 10 | 7274000475/262144 | 513157/67108864 |
| 11 | -753426694125/1048576 | -53151795/2952790016 |
| 12 | 3490543811475/4194304 | 82082159/47244640256 |
| 13 | -834403776221925/16777216 | -1509345229/188978561024 |
| 14 | -19221788273793525/67108864 | -64573068863/19653770346496 |
| 15 | -717789378292804125/268435456 | -14614054443/7146825580544 |
| 16 | 31723568220115303875/1073741824 | 7104743589511/5031365208702976 |
| 17 | -140087143112528863125/4294967296 | -1845507165665/20125460834811904 |
| 18 | -182964894818522014477125/17179869184 | -413903362322923/248823879412219904 |
| 19 | -1887835402113443747362125/68719476736 | -190191473005959/842173130318282752 |

### 21a â€” coefficients 0 through 19

| n | c_n | c_n/n! |
|---:|---:|---:|
| 0 | 1 | 1 |
| 1 | 1 | 1 |
| 2 | -905/3969 | -905/7938 |
| 3 | -1318/27783 | -659/83349 |
| 4 | -4357004/5250987 | -1089251/31505922 |
| 5 | 2121080/583443 | 53027/1750329 |
| 6 | -38714882308/6947055801 | -9678720577/1250470044180 |
| 7 | 4782446912/425329947 | 298902932/133978933305 |
| 8 | 346256788547680/3938980639167 | 2164104928423/992623121070084 |
| 9 | -3320127455840/437664515463 | -20750796599/992623121070084 |
| 10 | 193274504686264160/1737090461872647 | 1207965654289151/39397211675271633960 |
| 11 | -7759719497388864640/12159633233108529 | -12124561714670101/758396324748978953730 |
| 12 | -20101786476667389280448/2298170681057511981 | -314090413697927957507/17200428645306842670596400 |
| 13 | 2262149965884242737216/328310097293930283 | 35346093216941292769/31943653198426993531107600 |
| 14 | 28421381962967115741726400/1303062776159609293227 | 17763363726854447338579/70999241344951772901980996064 |
| 15 | 124355294277752051572902400/186151825165658470461 | 9715257365449379029133/19017653931683510598744909660 |
| 16 | 6012597624168961840025530941440/1723952052859163094939321 | 73395967091906272461249157/440306232653302479137441520903150 |
| 17 | 515021061149243584510723932160/21283358677273618456041 | 483606014450536719229571/7108457697156830147518049245350 |
| 18 | 193098873298338795222806402355625984/760262855310890924868240561 | 2946454975865765307965185582819/74271956089540695437101479349745599875 |
| 19 | -18257035541123100158257753293330698240/15965519961528709422233051781 | -46918779659547440784996282106627/4991075449217134733373219412302904311600 |

... (15 outputs left out)

| n | c_n | c_n/n! |
|---:|---:|---:|
| 0 | 1 | 1 |
| 1 | 1 | 1 |
| 2 | -4/5 | -2/5 |
| 3 | 314/125 | 157/375 |
| 4 | 31192/625 | 3899/1875 |
| 5 | -1977728/625 | -247216/9375 |
| 6 | 4428752/3125 | 276797/140625 |
| 7 | -4829523664/78125 | -43120747/3515625 |
| 8 | 346814303488/390625 | 2709486746/123046875 |
| 9 | -300766884736/78125 | -2349741287/221484375 |
| 10 | -38348182340864/390625 | -149797587269/5537109375 |
| 11 | -6801387461287936/48828125 | -2415265433696/692138671875 |
| 12 | -6368944313448915968/244140625 | -80774963390941/1483154296875 |
| 13 | -253011158115384660992/244140625 | -246834374722333/1483154296875 |
| 14 | 5961038888167461367808/1220703125 | 37800824930039198/674835205078125 |
| 15 | 23589809944455975265396736/30517578125 | 149590414116122002241/253063201904296875 |
| 16 | 365135918752925243199029248/152587890625 | 1013005811526004425601/8857212066650390625 |
| 17 | 4623285034520021238369746944/152587890625 | 754499798048706256784/8857212066650390625 |
| 18 | -1804346373610853177832359919616/762939453125 | -2502921882783165918291071/6775767230987548828125 |
| 19 | -35254756724197283893941957492736/19073486328125 | -2573899025379329399871239/169394180774688720703125 |

### 58b â€” coefficients 0 through 19

| n | c_n | c_n/n! |
|---:|---:|---:|
| 0 | 1 | 1 |
| 1 | 1 | 1 |
| 2 | 140/9 | 70/9 |
| 3 | 262/3 | 131/9 |
| 4 | -39592/81 | -4949/243 |
| 5 | 420224/81 | 52528/1215 |
| 6 | -303950320/729 | -3799379/6561 |
| 7 | -519149456/729 | -4635263/32805 |
| 8 | 482738922752/6561 | 3771397834/2066715 |
| 9 | 475051014784/729 | 3711336053/2066715 |
| 10 | 1429957306131200/59049 | 31918689869/4782969 |
| 11 | 5467784308760576/59049 | 1941684768736/837019575 |
| 12 | 18433309687885503488/531441 | 233782844052931/3228504075 |
| 13 | 422815682063031921664/531441 | 412493446068611/3228504075 |
| 14 | -33489133317188465889280/4782969 | -42473028253333586/528828967485 |
| 15 | -2063463017938102093637632/1594323 | -13085068853605050817/13220724187125 |
| 16 | 670068453560967521041743872/43046721 | 1858987852785887343089/2498716871366625 |
| 17 | 147020457335422648774010863616/43046721 | 23993092474368930555376/2498716871366625 |
| 18 | -41119409848227681277846268477440/387420489 | -1629694548697639813226329/98306660910624075 |
| 19 | -808116333331724112619144964931584/387420489 | -58999409896027349635887691/3440733131871842625 |

PASS: all twelve printed expansions; 32 coefficients computed and 30 residual coefficients checked per form.


The exact checks certify the stated orders, centers and consequences of the
recurrence input. The numerical scale comparisons do not prove the algebraic
recognition of the scale. AI assistance and GPL-3.0 terms are stated in README.